In [76]:
import sys
import os
sys.path.append(os.path.abspath('../../../'))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from functions.preproc import *
from functions.ucphm import *
from functions.new_variables import *
from functions.ensembles import *
from scipy.stats import skew, shapiro

In [77]:
dataset = pd.read_excel('../../../BreastCareCond.xlsx')
data_code = pd.read_excel('../../../Codificação variaveis.xlsx')

Using the variables she used

This also as the surgery data, the only data that appeared here were this three variables 'Type of 1st intervention', 'Axillary operation', 'SLN procedure'

In [78]:
#Defining the proposed has good treatment variables to use (the empty ones were removed)
relevant_variables = ['PERSON_ID', 'Regional LN recurrence', 'Date of 1st regional recurrence', 'EBC', 'Survival days']
numerical_variables = ['Age', 'Ki67 positive cells (%) at CB']
categorical_variables = ['N (Regional nodes affected)', 'Side location of the lesion', 'Type of invasion at CB', 
                         'Isotype at CB', 
                         'Oestrogen receptor status at CB', 'Progesterone receptor status at CB', 
                         'Grade at CB', 'Her2 overexpression (with immunohystochemistry) at CB']
                        
date_columns = ['Date of CB ', 'Date of 1st breast operation', 'Date of intervention on sentinel lymph node',
                'Date of intervention (axilla)']

In [79]:
#correcting th error in the column name
dataset.rename(columns={'Istotype at CB': 'Isotype at CB'}, inplace=True)

In [80]:
dataset = date_of_diagnosis(dataset, date_columns)

relevant_dates = ['Date of diagnosis', 'Date of 1st regional recurrence']
dataset = remove_zeros_na(dataset, relevant_dates)
dataset = format_date(dataset, relevant_dates)

dataset = survival_days(dataset, ['Date of diagnosis'], ['Date of 1st regional recurrence'])
dataset = tte_negative_patients(dataset)
dataset = filter_EBC(dataset)
dataset.reset_index(drop=True, inplace=True)

In [81]:
# Find duplicated patients
mask = (dataset.duplicated(subset='PERSON_ID', keep=False) & (dataset['Regional LN recurrence'] == 1))

# For each duplicated PERSON_ID with recurrence, keep only the first row
to_remove = (
    dataset[mask]
    .groupby('PERSON_ID')
    .tail(-1)
    .index
)

dataset = dataset.drop(to_remove)

In [82]:
#since the clinical staging uses the nodes
node = ['N (Regional nodes affected)']
fill_with_mode(dataset, node)

'Done'

In [83]:
change_to_numeric(dataset, numerical_variables)
change_to_categorical(dataset, categorical_variables)

Done
Done


In [84]:
columns_to_map = {
    'Side location of the lesion': {'L': 0, 'R': 1},
    'Type of invasion at CB': {1: 0, 2: 1},
    'Oestrogen receptor status at CB': {1: 0, 2: 1},
    'Progesterone receptor status at CB': {1: 0, 2: 1},
    'N (Regional nodes affected)': {0: 0, 1: 1, 2: 2_3, 3: 2_3}
}

for column, mapping in columns_to_map.items():
    dataset[column] = dataset[column].map(mapping)

In [85]:
mode_variables = ['Side location of the lesion']
fill_with_mode(dataset, mode_variables)

'Done'

In [86]:
# ---------------------------------------------------------------------------
# Diagnostic pathology block: MICE instead of mode / missing-indicator encoding.
#
# This replaces `one_hot_encode_with_missing_indicator(PR, Grade, HER2)`. Two reasons
# it was changed (both established in the missingness analysis in new_data.ipynb):
#
#   1. The missingness indicators are perfectly NESTED -- ER missing is a subset of
#      PR missing is a subset of HER2 missing, with zero violations. Encoding them as
#      separate dummies therefore spends 3 degrees of freedom to carry ~1 df of
#      information and produces a near-singular design; the fitted Cox coefficients
#      came out degenerate (one hazard ratio at 4e7, another at 0.00, CIs [0, inf]).
#      A likelihood-ratio test showed the encoding added nothing to recurrence
#      prediction (chi2 = 1.49 on 3 df, p = 0.69) -- an expensive way to buy no signal
#      from a dataset with only ~35 events.
#
#   2. Mode imputation is not a safe fallback either. Among patients where both are
#      recorded, ER predicts PR with 96.2% concordance, and a few hundred patients
#      have PR missing while ER *is* recorded. Blind mode imputation overrides their
#      own ER status and manufactures receptor combinations that mostly do not occur.
#      Chained equations use ER (and grade, isotype, type of invasion) to impute PR.
#
# The `_year` / `_cumhaz` / `_event` columns are auxiliary predictors, used by the
# imputation model and dropped immediately afterwards:
#   _year    -- MICE assumes MAR, i.e. missingness is random *conditional on the
#               variables in the imputation model*. Missingness here depends strongly
#               on year of diagnosis, so leaving year out would violate the very
#               assumption the method rests on.
#   _cumhaz  -- Nelson-Aalen cumulative hazard, plus the event indicator. Including
#     _event    the outcome when imputing covariates is standard for survival data
#               (White & Royston, 2009); omitting it biases the covariate-outcome
#               association toward the null.
# ---------------------------------------------------------------------------
from lifelines import NelsonAalenFitter

pathology_categorical = ['Oestrogen receptor status at CB', 'Progesterone receptor status at CB',
                         'Her2 overexpression (with immunohystochemistry) at CB', 'Grade at CB',
                         'Isotype at CB', 'Type of invasion at CB']
pathology_continuous = ['Ki67 positive cells (%) at CB']

dataset['_year'] = pd.to_datetime(dataset['Date of diagnosis']).dt.year
_naf = NelsonAalenFitter().fit(dataset['Survival days'], dataset['Regional LN recurrence'])
dataset['_cumhaz'] = _naf.cumulative_hazard_at_times(dataset['Survival days']).values
dataset['_event'] = dataset['Regional LN recurrence'].astype(float)
_aux = ['_year', '_cumhaz', '_event']

print('Missing before MICE:')
print(dataset[pathology_categorical + pathology_continuous].isna().sum().to_string())

# m=10 completed datasets: the first is used downstream, the rest quantify how much
# the imputation is guessing (see the FMI table below).
_imputations = mice_impute(dataset, pathology_categorical, pathology_continuous,
                           aux_variables=_aux, n_iter=5, m=10, random_state=42,
                           return_all=True)
dataset = _imputations[0].drop(columns=_aux)

print('\nMissing after MICE:')
print(dataset[pathology_categorical + pathology_continuous].isna().sum().to_string())

# Fraction of missing information: how much the imputed values disagree across the 10
# runs. Low values mean a single completed dataset loses little relative to pooling;
# report this rather than assuming it. Values well above ~0.1 (HER2 here) are the ones
# to mention as a limitation, since a single imputation understates their uncertainty.
print('\nImputation stability across the 10 runs:')
print(imputation_fmi(_imputations, pathology_categorical).to_string(index=False))


Missing before MICE:
Oestrogen receptor status at CB                          136
Progesterone receptor status at CB                       445
Her2 overexpression (with immunohystochemistry) at CB    475
Grade at CB                                              194
Isotype at CB                                            142
Type of invasion at CB                                   141
Ki67 positive cells (%) at CB                            536



Missing after MICE:
Oestrogen receptor status at CB                          0
Progesterone receptor status at CB                       0
Her2 overexpression (with immunohystochemistry) at CB    0
Grade at CB                                              0
Isotype at CB                                            0
Type of invasion at CB                                   0
Ki67 positive cells (%) at CB                            0

Imputation stability across the 10 runs:
                                             variable  between_imputation_var  total_var  FMI_approx
                      Oestrogen receptor status at CB                 0.00577    0.12342      0.0467
                   Progesterone receptor status at CB                 0.02109    0.18563      0.1136
Her2 overexpression (with immunohystochemistry) at CB                 0.07683    0.47815      0.1607
                                          Grade at CB                 0.01693    0.37604      0.0450
                   

In [87]:
# Grade and HER2 join the ordinary one-hot list: they no longer carry a '404' missing
# level, so they encode like any other categorical (drop_first, k-1 dummies). ER, PR
# and Type of invasion stay as single binary columns -- they were remapped to 0/1 in
# the cell above, and one-hot encoding a binary variable would only duplicate it.
one_hot_variables = ['N (Regional nodes affected)',
                     'Isotype at CB', 
                     'Grade at CB',
                     'Her2 overexpression (with immunohystochemistry) at CB']
ohe_dataset, new_column_names = one_hot_encoding(dataset, one_hot_variables, drop_first='first')

In [88]:
categorical_variables = [col for col in categorical_variables if col not in one_hot_variables]
categorical_variables.extend(new_column_names.tolist())

selected_variables = relevant_variables + numerical_variables + categorical_variables

In [89]:
treatment_variables = ['Radiotherapy (RT) performed', 'Conventional RT fraction', 'Total administered dose', 'Boost dose administered', 
                       'Radiotherapy on chest wall', 'Radiotherapy on supraclavicular area', 'Radiotherapy on internal mammary chain',
                         'State (if hormone therapy prescribed)', 'Treatment in association with chemotherapy', 
                        'Chemotherapy performed', 'Adjuvant chemotherapy', 'Adjuvant therapy with biological drugs', 'Endocrine therapy performed']
selected_variables_treat = selected_variables + treatment_variables

survival_dataset = ohe_dataset[selected_variables_treat].copy()

In [90]:
numerical_data, categorical_data = type_separation(data_code, treatment_variables)
len(numerical_data), len(categorical_data)

(3, 10)

In [91]:
dataset = survival_dataset.copy()

In [92]:
replace_missing_values(dataset, numerical_data, 0)

Done


In [93]:
dataset.reset_index(inplace=True, drop=True)
replace_missing_values(dataset, categorical_data, 0)

Done


In [94]:
variable_two_to_zero = ['Adjuvant chemotherapy', 'Chemotherapy performed', 'Endocrine therapy performed']
two_to_zero(dataset, variable_two_to_zero)

bio_variable = 'Adjuvant therapy with biological drugs'
dataset = encode_bio_therapy(dataset, bio_variable)

In [95]:
feature_columns = ['Adjuvant therapy using biological drugs'] 
feature_columns = feature_columns + categorical_data + numerical_data

treatment_variables = [col for col in feature_columns if col not in bio_variable]

In [96]:
# Create a derived neojuven therapy only variable.
# It is 1 when adjuvant therapy with biological drugs is 0 and endocrine therapy performed is 1.

dataset['Neoadjuvant (Only) therapy with biological drugs'] = (
    (dataset['Adjuvant therapy using biological drugs'] == 0) &
    (dataset['Endocrine therapy performed'] == 1)
).astype(int)

treatment_variables = [
    col for col in treatment_variables
    if col != 'Endocrine therapy performed'
]
treatment_variables.append('Neoadjuvant (Only) therapy with biological drugs')

In [97]:
to_remove = ['PERSON_ID', 'Regional LN recurrence', 'Date of 1st regional recurrence', 'EBC', 'Survival days']
important_variables = ['PERSON_ID', 'Regional LN recurrence', 'Survival days']
diagnosis_variables = [col for col in selected_variables if col not in to_remove]
selected_columns = important_variables + treatment_variables + diagnosis_variables

In [98]:
survival_dataset = dataset[selected_columns]

In [99]:
# Check if there are any NaN values in the dataset
contains_nan = survival_dataset.isnull().any().any()
print("Does survival_dataset contain NaN values?", contains_nan)

# Optionally, to see which columns contain NaN values
nan_columns = survival_dataset.columns[survival_dataset.isnull().any()].tolist()
print("Columns with NaN values:", nan_columns)

Does survival_dataset contain NaN values? False
Columns with NaN values: []


In [100]:
survival_dataset.to_csv("data.csv")

In [101]:
with open("diagnosis.txt", "w") as file:
    for item in diagnosis_variables:
        file.write(str(item) + "\n")
with open("treatment.txt", "w") as file:
    for item in treatment_variables:
        file.write(str(item) + "\n")